# Ponderação das Edições

In [6]:
import pandas as pd

In [23]:
arquivo_base = "Base_de_Dados_Pesquisa_Gestao_Municipal.xlsx"
arquivo_populacao = "populacao_percent_2022.csv"

df = pd.read_excel(arquivo_base)
populacao_percent = pd.read_csv(arquivo_populacao)

print("Base:", df.shape)
print("População:", populacao_percent.shape)

Base: (21762, 15)
População: (60, 5)


In [24]:
populacao_percent = populacao_percent.rename(columns={"sexo": "Sexo", "idade": "Faixa etária", "raca": "Cor/raça"})
for coluna in ["Sexo", "Faixa etária", "Cor/raça"]:
    df[coluna] = df[coluna].astype(str).str.strip()
    populacao_percent[coluna] = (populacao_percent[coluna].astype(str).str.strip())
    
mapa_idade = {
    "16 a 24 anos": "16-24",
    "25 a 29 anos": "25-29",
    "30 a 39 anos": "30-39",
    "40 a 49 anos": "40-49",
    "50 a 59 anos": "50-59",
    "60 anos ou mais": "60+"
}

populacao_percent["Faixa etária"] = (
    populacao_percent["Faixa etária"]
    .replace(mapa_idade)
)
populacao_percent["pop_percent"] = pd.to_numeric(populacao_percent["pop_percent"], errors = "coerce")

In [ ]:
def calcular_pesos_edicao(df_edicao, populacao_percent):
    amostra = df_edicao.groupby(["Sexo", "Faixa etária", "Cor/raça"]).size().reset_index(name="freq")
    amostra["amostra_percent"] = amostra["freq"] / amostra["freq"].sum()
    full_df = amostra.merge(populacao_percent[["Sexo", "Cor/raça", "Faixa etária", "pop_percent"]], on=["Sexo", "Cor/raça", "Faixa etária"], how="left")
    if full_df["pop_percent"].isna().any():
        faltantes = full_df[full_df["pop_percent"].isna()][["Sexo", "Cor/raça", "Faixa etária"]]
        print("Combinações sem correspondência no arquivo de população:")
        print(faltantes.to_string(index=False))
    full_df["peso"] = full_df["pop_percent"] / full_df["amostra_percent"]
    full_df.loc[full_df["peso"] == 0, "peso"] = 0.001
    return full_df[["Sexo", "Cor/raça", "Faixa etária", "peso"]]

df_com_peso = []
edicoes = sorted(df["Edição"].dropna().unique(), reverse=True)

for edicao in edicoes:
    print(f"\nCalculando ponderação da edição {edicao}")
    df_edicao = df[df["Edição"] == edicao].copy()
    pesos = calcular_pesos_edicao(df_edicao, populacao_percent)
    df_edicao = df_edicao.merge(pesos, on=["Sexo", "Cor/raça", "Faixa etária"], how="left")
    df_com_peso.append(df_edicao)

df_final = pd.concat(df_com_peso, ignore_index=True)


Calculando ponderação da edição 16

Calculando ponderação da edição 15

Calculando ponderação da edição 14

Calculando ponderação da edição 13

Calculando ponderação da edição 12

Calculando ponderação da edição 11

Calculando ponderação da edição 10

Calculando ponderação da edição 9

Calculando ponderação da edição 8

Calculando ponderação da edição 7

Calculando ponderação da edição 6

Calculando ponderação da edição 5

Calculando ponderação da edição 4

Calculando ponderação da edição 3

Calculando ponderação da edição 2

Calculando ponderação da edição 1


In [40]:
arquivo_saida = "Base_de_Dados_Pesquisa_Gestao_Municipal_ponderada.xlsx"

df_final.to_excel(
    arquivo_saida,
    index=False
)

print(f"Arquivo salvo: {arquivo_saida}")

Arquivo salvo: Base_de_Dados_Pesquisa_Gestao_Municipal_ponderada.xlsx
